This cell defines a **Tkinter-based interactive dashboard** (`FableDashboard`) for visualizing scenario data from Excel files.  
It allows loading two scenario files, reading their **GHG emissions, production, trade balance, and jobs data**, and generating interactive **Plotly charts** and **CSV exports**. The dashboard also handles session management, logging, and temporary storage.  

Currently, it supports **GHG, Production, Trade, and Jobs** analysis, but in the future, it should also integrate additional sheets for **LAND, BIODIVERSITY, FOOD, and N/P (nutrients)** to expand the scope of scenario evaluation.


In [2]:
import os
from pathlib import Path
from datetime import datetime
import logging
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import shutil
import tempfile

REQUIRED_SECTOR_COLUMNS = [
    "Calc CO2e from livestock",
    "Total calc. CO2e from crops",
    "Calc CO2 from deforestation",
    "Calc CO2 sequestration",
]

class FableDashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("FABLE Scenario Dashboard")

        # Session directories and logging
        self.session_tmp_dir = Path(tempfile.mkdtemp(prefix="fable_dashboard_"))
        self.exports_dir = None  # set after selecting files (next to inputs)
        self._configure_logging()

        # Scenario state and caches
        self.scenarios = []  # [{label, source_path, temp_path, selections, ghg_df}]
        self.combined_cache = None
        self.production_cache = None
        self.trade_cache = None

        # Visualization config
        self.cols_to_plot = REQUIRED_SECTOR_COLUMNS.copy()
        self.scenario_colors = {"Scenario A": "blue", "Scenario B": "red"}
        self.sector_colors = {
            "Calc CO2e from livestock": "#636EFA",
            "Total calc. CO2e from crops": "#EF553B",
            "Calc CO2 from deforestation": "#00CC96",
            "Calc CO2 sequestration": "#AB63FA",
        }
        # Production config
        self.production_cols = ["prodq_hist", "prodq_feas", "prodq_targ"]
        self.production_colors = {
            "prodq_hist": "#1f77b4",
            "prodq_feas": "#ff7f0e",
            "prodq_targ": "#2ca02c",
        }

        self.create_widgets()
        self._set_buttons_state("disabled")
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)

    def _configure_logging(self):
        try:
            log_path = self.session_tmp_dir / "fable_dashboard.log"
            logging.basicConfig(
                filename=str(log_path),
                filemode="a",
                level=logging.INFO,
                format="%(asctime)s %(levelname)s %(message)s",
            )
            logging.info("Session started. Temp dir: %s", self.session_tmp_dir)
        except Exception as e:
            logging.basicConfig(level=logging.INFO)
            logging.exception("Failed to configure file logging: %s", e)

    def create_widgets(self):
        tk.Label(self.root, text="Select Scenario A and B Files:").grid(row=0, column=0, padx=10, pady=5, sticky="w")
        tk.Button(self.root, text="Select Scenario Files", command=self.select_scenarios).grid(row=0, column=1, padx=5)

        self.visual_frame = tk.LabelFrame(self.root, text="Visualizations", padx=10, pady=10)
        self.visual_frame.grid(row=1, column=0, columnspan=2, padx=10, pady=10, sticky="ew")
        self.btn_bar = tk.Button(self.visual_frame, text="Yearly GHG by Sector", command=self.show_bar_chart)
        self.btn_line = tk.Button(self.visual_frame, text="Total GHG Line Chart", command=self.show_total_ghg_chart)
        self.btn_export_csv = tk.Button(self.visual_frame, text="Export GHG CSV", command=self.export_combined_csv)
        # Production buttons
        self.btn_prod_bar = tk.Button(self.visual_frame, text="Production by Series", command=self.show_production_bar)
        self.btn_prod_export = tk.Button(self.visual_frame, text="Export Production CSV", command=self.export_production_csv)
        # Trade balance
        self.btn_trade = tk.Button(self.visual_frame, text="Trade Balance", command=self.show_trade_balance)
        # Jobs buttons
        self.btn_jobs_pct = tk.Button(self.visual_frame, text="Jobs % of FTE", command=self.show_jobs_percent)
        self.btn_jobs_fte = tk.Button(self.visual_frame, text="FTE Workers (stacked)", command=self.show_jobs_fte_stacked)

        self.btn_bar.pack(pady=5)
        self.btn_line.pack(pady=5)
        self.btn_export_csv.pack(pady=5)
        self.btn_prod_bar.pack(pady=5)
        self.btn_prod_export.pack(pady=5)
        self.btn_trade.pack(pady=5)
        self.btn_jobs_pct.pack(pady=5)
        self.btn_jobs_fte.pack(pady=5)

        self.summary_text = tk.Text(self.root, width=100, height=10, state="disabled")
        self.summary_text.grid(row=2, column=0, columnspan=2, padx=10, pady=10)

        self.status_var = tk.StringVar(value="Select two scenario files to begin…")
        self.status_label = tk.Label(self.root, textvariable=self.status_var, anchor="w")
        self.status_label.grid(row=3, column=0, columnspan=2, sticky="ew", padx=10, pady=(0,10))

    def _set_buttons_state(self, state: str):
        for b in (self.btn_bar, self.btn_line, self.btn_export_csv,
                  self.btn_prod_bar, self.btn_prod_export, self.btn_trade,
                  self.btn_jobs_pct, self.btn_jobs_fte):
            b.config(state=state)

    # --- Helpers: Excel parsing ---
    def _resolve_sheet_name(self, excel_path: Path, tokens) -> str:
        xls = pd.ExcelFile(excel_path)
        def norm(s):
            return " ".join(str(s).strip().lower().split())
        for name in xls.sheet_names:
            n = norm(name)
            if all(tok in n for tok in tokens):
                return name
        raise ValueError(f"Worksheet matching tokens {tokens} not found.")

    def _read_ghg_sheet(self, excel_path: Path) -> pd.DataFrame:
        ghg_sheet = self._resolve_sheet_name(excel_path, ["ghg"])  # case/spacing tolerant
        sample = pd.read_excel(excel_path, sheet_name=ghg_sheet, header=None, nrows=80)
        year_row = next((i for i in range(len(sample)) if sample.iloc[i].astype(str).str.strip().str.lower().eq("year").any()), None)
        if year_row is None:
            raise ValueError("Could not find header row containing 'Year' in GHG sheet.")
        df = pd.read_excel(excel_path, sheet_name=ghg_sheet, skiprows=year_row)
        if "Year" not in df.columns:
            raise ValueError("GHG sheet missing 'Year' column after header detection.")
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df = df[df["Year"].between(2000, 2050)]
        df = df.dropna(subset=["Year"]).copy()
        df["Year"] = df["Year"].astype(int)
        for c in df.columns:
            if c != "Year":
                df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    def _read_selections(self, excel_path: Path) -> dict:
        # tolerate variants like "SCENAROS selection" and casing/spacing
        sel_sheet = self._resolve_sheet_name(excel_path, ["scenar", "select"])
        df = pd.read_excel(excel_path, sheet_name=sel_sheet, header=None)
        selections = {}
        current_table = None
        for _, row in df.iterrows():
            first = str(row[0]).strip() if row[0] is not None else ""
            second = str(row[1]).strip() if len(row) > 1 and row[1] is not None else ""
            if first.startswith("TABLE:"):
                current_table = first.replace("TABLE:", "").strip()
                selections[current_table] = []
                continue
            if first.lower() == "x" and current_table:
                selections[current_table].append(second)
        return selections

    def _read_jobs_sheet(self, excel_path: Path) -> dict:
        jobs_sheet = self._resolve_sheet_name(excel_path, ["jobs"])  # JOBS sheet
        # Exact ranges provided by user
        years = pd.read_excel(excel_path, sheet_name=jobs_sheet, header=None, usecols="K", skiprows=14, nrows=11).squeeze("columns")
        pct = pd.read_excel(excel_path, sheet_name=jobs_sheet, header=None, usecols="P", skiprows=14, nrows=11).squeeze("columns")
        jobs_crop = pd.read_excel(excel_path, sheet_name=jobs_sheet, header=None, usecols="L", skiprows=14, nrows=11).squeeze("columns")
        jobs_livestock = pd.read_excel(excel_path, sheet_name=jobs_sheet, header=None, usecols="M", skiprows=14, nrows=11).squeeze("columns")
        df_pct = pd.DataFrame({"Year": years, "calc % of on farm jobs in FTE": pct})
        df_fte = pd.DataFrame({"Year": years, "jobs in crop": jobs_crop, "jobs in livestock": jobs_livestock})
        for c in ["Year"]:
            df_pct[c] = pd.to_numeric(df_pct[c], errors="coerce")
            df_fte[c] = pd.to_numeric(df_fte[c], errors="coerce")
        df_pct = df_pct.dropna(subset=["Year"]).copy()
        df_fte = df_fte.dropna(subset=["Year"]).copy()
        df_pct["Year"] = df_pct["Year"].astype("Int64")
        df_fte["Year"] = df_fte["Year"].astype("Int64")
        # numeric coercion
        df_pct["calc % of on farm jobs in FTE"] = pd.to_numeric(df_pct["calc % of on farm jobs in FTE"], errors="coerce")
        df_fte["jobs in crop"] = pd.to_numeric(df_fte["jobs in crop"], errors="coerce")
        df_fte["jobs in livestock"] = pd.to_numeric(df_fte["jobs in livestock"], errors="coerce")
        return {"pct": df_pct, "fte": df_fte}

    def _read_trade_sheet(self, excel_path: Path) -> pd.DataFrame:
        trade_sheet = self._resolve_sheet_name(excel_path, ["trade"])  # TRADE sheet
        # Use exact chart source ranges: Years K14:K24, Values O14:O24
        years = pd.read_excel(
            excel_path,
            sheet_name=trade_sheet,
            header=None,
            usecols="K",
            skiprows=13,  # start at row 14 (1-based)
            nrows=11,
        ).squeeze("columns")
        values = pd.read_excel(
            excel_path,
            sheet_name=trade_sheet,
            header=None,
            usecols="O",
            skiprows=13,
            nrows=11,
        ).squeeze("columns")
        df = pd.DataFrame({"Year": years, "Trade balance": values})
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
        df["Trade balance"] = pd.to_numeric(df["Trade balance"], errors="coerce")
        df = df.dropna(subset=["Year"]).copy()
        return df

    def _read_production_sheet(self, excel_path: Path) -> pd.DataFrame:
        # Find PRODUCTION sheet case-insensitively
        prod_sheet = self._resolve_sheet_name(excel_path, ["product"])  # matches 'production'
        # Read a larger sample without headers to detect the header row
        sample = pd.read_excel(excel_path, sheet_name=prod_sheet, header=None, nrows=400)
        def norm(s):
            return str(s).strip().lower()
        header_row = None
        for i in range(len(sample)):
            row_vals = sample.iloc[i].tolist()
            row_norm = [norm(v) for v in row_vals]
            # Detect typical pivot headers like 'Row Labels' + 'sum of prodq_*'
            has_rowlabels = any("row labels" in v for v in row_norm)
            has_prodq = any("prodq" in v for v in row_norm)
            has_year = any(v == "year" for v in row_norm)
            if (has_rowlabels and has_prodq) or has_year:
                header_row = i
                break
        if header_row is None:
            # fallback near the hint B59 (0-based index 58)
            header_row = 58
        df = pd.read_excel(excel_path, sheet_name=prod_sheet, skiprows=header_row)
        # Flatten MultiIndex headers if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [" ".join([str(x) for x in tup if str(x) != "nan"]).strip() for tup in df.columns.values]
        # Normalize first column to 'Year'
        first_col = df.columns[0]
        if norm(first_col) in ("row labels", "year"):
            df = df.rename(columns={first_col: "Year"})
        if "Year" not in df.columns:
            raise ValueError("Could not detect 'Year' column in PRODUCTION sheet.")
        # Drop totals rows
        df = df[~df["Year"].astype(str).str.contains("total", case=False, na=False)]
        # Coerce types
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df = df.dropna(subset=["Year"]).copy()
        df["Year"] = df["Year"].astype(int)
        # Coerce production numeric columns and auto-detect production series
        detected_cols = []
        for c in df.columns:
            if c != "Year":
                df[c] = pd.to_numeric(df[c], errors="coerce")
                if "prodq" in norm(c):
                    detected_cols.append(c)
        # If we detected series, update config; keep previous otherwise
        if detected_cols:
            self.production_cols = detected_cols
            logging.info("Production columns detected: %s | headers: %s", self.production_cols, list(df.columns))
            return df
        # Hardcoded fallback: B60:E70 => Year, prodq_hist, prodq_feas, prodq_targ
        logging.info("Falling back to hardcoded PRODUCTION range B60:E70")
        df_fixed = pd.read_excel(
            excel_path,
            sheet_name=prod_sheet,
            header=None,
            usecols="B:E",
            skiprows=59,  # start at row 60 (0-based index)
            nrows=11,     # rows 60..70 inclusive
        )
        df_fixed.columns = ["Year", "prodq_hist", "prodq_feas", "prodq_targ"]
        df_fixed["Year"] = pd.to_numeric(df_fixed["Year"], errors="coerce").astype("Int64")
        for c in ["prodq_hist", "prodq_feas", "prodq_targ"]:
            df_fixed[c] = pd.to_numeric(df_fixed[c], errors="coerce")
        df_fixed = df_fixed.dropna(subset=["Year"]).copy()
        self.production_cols = ["prodq_hist", "prodq_feas", "prodq_targ"]
        logging.info("Production fallback rows: %s", df_fixed.shape[0])
        return df_fixed

    # --- File selection ---
    def select_scenarios(self):
        try:
            paths = filedialog.askopenfilenames(title="Select Scenario A then B", filetypes=[("Excel files", "*.xlsx *.xlsm *.xls")])
            if len(paths) != 2:
                messagebox.showerror("Error", "Please select exactly two scenario files.")
                return

            first_dir = Path(paths[0]).parent
            self.exports_dir = first_dir / "exports"
            self.exports_dir.mkdir(exist_ok=True)

            self.scenarios = []
            for idx, p in enumerate(paths):
                src = Path(p)
                label = f"Scenario_{src.stem}"
                dst = self.session_tmp_dir / src.name
                shutil.copy(src, dst)
                selections = self._read_selections(dst)
                ghg_df = self._read_ghg_sheet(dst)
                prod_df = self._read_production_sheet(dst)
                trade_df = self._read_trade_sheet(dst)
                jobs_data = self._read_jobs_sheet(dst)
                self.scenarios.append({
                    "label": label,
                    "source_path": src,
                    "temp_path": dst,
                    "selections": selections,
                    "ghg_df": ghg_df,
                    "prod_df": prod_df,
                    "trade_df": trade_df,
                    "jobs_pct": jobs_data["pct"],
                    "jobs_fte": jobs_data["fte"],
                })

            # dynamic scenario color mapping (first blue, second red)
            if len(self.scenarios) == 2:
                self.scenario_colors = {
                    self.scenarios[0]["label"]: "blue",
                    self.scenarios[1]["label"]: "red",
                }

            self.combined_cache = None
            self.production_cache = None
            self.trade_cache = None
            self._set_buttons_state("normal")
            self.update_summary()
            messagebox.showinfo("Loaded", "Scenario files loaded and validated successfully!")
            self.status_var.set("Scenarios loaded. Ready to visualize.")
            logging.info("Loaded scenarios: %s and %s", paths[0], paths[1])
        except Exception as e:
            logging.exception("Failed to load scenarios: %s", e)
            self._set_buttons_state("disabled")
            self.scenarios = []
            self.combined_cache = None
            messagebox.showerror("Load Error", f"Failed to load scenarios.\n{e}")
            self.status_var.set("Load failed. See log for details.")

    # --- Scenario summary ---
    def update_summary(self):
        self.summary_text.config(state="normal")
        self.summary_text.delete("1.0", tk.END)
        if not self.scenarios:
            self.summary_text.insert(tk.END, "No scenarios loaded.\n")
        for sc in self.scenarios:
            label = sc["label"]
            path_name = sc["source_path"].name
            self.summary_text.insert(tk.END, f"{label} ({path_name}):\n")
            selections = sc.get("selections", {})
            if selections:
                for table, items in selections.items():
                    if not items:
                        continue
                    for item in items:
                        self.summary_text.insert(tk.END, f"  {table}: {item}\n")
            else:
                self.summary_text.insert(tk.END, "  Could not read scenario selections\n")
            self.summary_text.insert(tk.END, "\n")
        self.summary_text.config(state="disabled")

    # --- Data prep (cached) ---
    def prepare_combined_data(self) -> pd.DataFrame:
        if self.combined_cache is not None:
            return self.combined_cache
        if len(self.scenarios) != 2:
            raise RuntimeError("Two scenarios must be loaded.")
        dfA = self.scenarios[0]["ghg_df"].copy()
        dfB = self.scenarios[1]["ghg_df"].copy()
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        cols_available = [c for c in self.cols_to_plot if c in set(dfA.columns) | set(dfB.columns)]
        def to_long(df, label):
            if not cols_available:
                return pd.DataFrame(columns=["Year", "Category", "MtCO2e/yr", "Scenario"])  
            long_df = df.melt(
                id_vars="Year",
                value_vars=[c for c in cols_available if c in df.columns],
                var_name="Category",
                value_name="MtCO2e/yr",
            )
            long_df["Scenario"] = label
            return long_df
        combined = pd.concat([to_long(dfA, labelA), to_long(dfB, labelB)], ignore_index=True)
        if not combined.empty:
            totals = combined.groupby(["Scenario", "Year"], as_index=False)["MtCO2e/yr"].sum().rename(columns={"MtCO2e/yr": "Total GHG"})
            combined = combined.merge(totals, on=["Scenario", "Year"], how="left")
        self.combined_cache = combined
        return combined

    # --- Helper to save outputs ---
    def _timestamped_name(self, base: str, ext: str) -> str:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{base}_{ts}.{ext}"

    def save_html(self, fig, base_name: str):
        if self.exports_dir is None:
            messagebox.showwarning("Warning", "No exports directory available.")
            return
        out_path = self.exports_dir / self._timestamped_name(base_name, "html")
        fig.write_html(str(out_path))
        messagebox.showinfo("Saved", f"Interactive HTML saved to:\n{out_path}")
        self.status_var.set(f"Saved: {out_path}")

    def _build_production_long(self) -> pd.DataFrame:
        if self.production_cache is not None:
            return self.production_cache
        if len(self.scenarios) != 2:
            raise RuntimeError("Two scenarios must be loaded.")
        dfA = self.scenarios[0]["prod_df"].copy()
        dfB = self.scenarios[1]["prod_df"].copy()
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]

        def norm(s):
            return str(s).strip().lower()
        def clean_series(col):
            c = norm(col)
            # remove common prefixes and spaces
            c = c.replace("sum of ", "").replace(" ", "")
            return c
        # detect prodq columns in each df, normalize header names
        colsA = [c for c in dfA.columns if "prodq" in norm(c)]
        colsB = [c for c in dfB.columns if "prodq" in norm(c)]
        if not colsA and not colsB:
            logging.warning("No prodq columns found. Headers A: %s | Headers B: %s", list(dfA.columns), list(dfB.columns))
            messagebox.showwarning("Warning", "No production columns (prodq_*) found. Check the PRODUCTION sheet headers.")
            return pd.DataFrame()
        # apply header normalization mapping
        renameA = {c: clean_series(c) for c in colsA}
        renameB = {c: clean_series(c) for c in colsB}
        dfA = dfA.rename(columns=renameA)
        dfB = dfB.rename(columns=renameB)
        # union of detected normalized columns across both
        cols_available = sorted(set(renameA.values()) | set(renameB.values()))
        # build long format with standardized Series names
        def to_long(df, label):
            present = [c for c in cols_available if c in df.columns]
            if not present:
                return pd.DataFrame(columns=["Year", "Series", "Value", "Scenario"])  
            long_df = df.melt(
                id_vars="Year",
                value_vars=present,
                var_name="Series",
                value_name="Value",
            )
            long_df["Scenario"] = label
            return long_df
        combined = pd.concat([to_long(dfA, labelA), to_long(dfB, labelB)], ignore_index=True)
        # update configured production columns/colors to normalized keys if needed
        self.production_cols = cols_available
        self.production_colors = {
            s: self.production_colors.get(s, self.production_colors.get(s.replace("sumof", ""), "#888888"))
            for s in self.production_cols
        }
        self.production_cache = combined
        return combined

    # --- Plotting ---
    def show_bar_chart(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        if df_combined.empty:
            messagebox.showwarning("Warning", "No sector columns found to plot.")
            return
        # Use source filenames in title
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        # Single plot: both scenarios in same axes; hashed pattern for second scenario
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        df_plot = df_combined.copy()
        df_plot["pattern"] = df_plot["Scenario"].map({labelA: "", labelB: "/"})
        fig = px.bar(
            df_plot,
            x="Year",
            y="MtCO2e/yr",
            color="Category",
            barmode="group",
            pattern_shape="Scenario",
            pattern_shape_map={labelA: "", labelB: "/"},
            color_discrete_map=self.sector_colors,
            title=f"Yearly GHG by Sector — {fileA} vs {fileB}",
        )
        fig.show()
        self.save_html(fig, "Yearly_GHG_by_Sector")

    def show_total_ghg_chart(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        if df_combined.empty:
            messagebox.showwarning("Warning", "No data available to plot totals.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        fig = px.line(
            df_combined.drop_duplicates(subset=["Year", "Scenario"]),
            x="Year",
            y="Total GHG",
            color="Scenario",
            color_discrete_map=self.scenario_colors,
            title=f"Total GHG (All Sectors) — {fileA} vs {fileB}",
        )
        fig.show()
        self.save_html(fig, "Total_GHG_Line")

    def show_combined_chart(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        if df_combined.empty:
            messagebox.showwarning("Warning", "No data available to plot.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=(f"Yearly GHG by Sector — {fileA} vs {fileB}", f"Total GHG — {fileA} vs {fileB}"))
        for scenario in (self.scenarios[0]["label"], self.scenarios[1]["label"]):
            df_s = df_combined[df_combined["Scenario"] == scenario]
            for category in self.cols_to_plot:
                df_cat = df_s[df_s["Category"] == category]
                if df_cat.empty:
                    continue
                fig.add_trace(
                    go.Bar(
                        x=df_cat["Year"],
                        y=df_cat["MtCO2e/yr"],
                        name=f"{category} ({scenario})",
                        marker_color=self.sector_colors.get(category, "#888888"),
                    ),
                    row=1,
                    col=1,
                )
            df_total = df_s.drop_duplicates(subset=["Year"])  
            fig.add_trace(
                go.Scatter(
                    x=df_total["Year"],
                    y=df_total["Total GHG"],
                    name=f"Total GHG ({scenario})",
                    line=dict(color=self.scenario_colors.get(scenario, "black"), width=3),
                ),
                row=2,
                col=1,
            )
        fig.update_layout(height=800, title_text="FABLE Scenario GHG Dashboard", barmode="group")
        fig.show()
        self.save_html(fig, "Combined_GHG_Charts")

    # --- Export combined CSV ---
    def export_combined_csv(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        if df_combined.empty:
            messagebox.showwarning("Warning", "No data available to export.")
            return
        out_path = self.exports_dir / self._timestamped_name("Combined_GHG_Data", "csv")
        df_combined.to_csv(out_path, index=False)
        messagebox.showinfo("Saved", f"CSV exported to:\n{out_path}")
        self.status_var.set(f"Saved: {out_path}")

    # --- Production charts/exports ---
    def show_production_bar(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_long = self._build_production_long()
        if df_long.empty:
            messagebox.showwarning("Warning", "No production data to plot.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        fig = px.bar(
            df_long,
            x="Year",
            y="Value",
            color="Series",
            barmode="group",
            pattern_shape="Scenario",
            pattern_shape_map={labelA: "", labelB: "/"},
            color_discrete_map=self.production_colors,
            title=f"Crop Production — {fileA} vs {fileB}",
        )
        fig.show()
        self.save_html(fig, "Production_by_Series")

    def show_production_total_line(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_long = self._build_production_long()
        if df_long.empty:
            messagebox.showwarning("Warning", "No production data to plot.")
            return
        totals = df_long.groupby(["Scenario", "Year"], as_index=False)["Value"].sum()
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        fig = px.line(
            totals,
            x="Year",
            y="Value",
            color="Scenario",
            color_discrete_map=self.scenario_colors,
            title=f"Total Crop Production — {fileA} vs {fileB}",
        )
        fig.show()
        self.save_html(fig, "Production_Total_Line")

    def export_production_csv(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_long = self._build_production_long()
        if df_long.empty:
            messagebox.showwarning("Warning", "No production data to export.")
            return
        out_path = self.exports_dir / self._timestamped_name("Production_Data", "csv")
        df_long.to_csv(out_path, index=False)
        messagebox.showinfo("Saved", f"CSV exported to:\n{out_path}")
        self.status_var.set(f"Saved: {out_path}")

    # --- Trade balance ---
    def show_trade_balance(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        def compute_trade(df: pd.DataFrame, label: str) -> pd.DataFrame:
            out = df[["Year", "Trade balance"]].copy()
            out["Scenario"] = label
            return out
        dfA = compute_trade(self.scenarios[0]["trade_df"], labelA)
        dfB = compute_trade(self.scenarios[1]["trade_df"], labelB)
        df_plot = pd.concat([dfA, dfB], ignore_index=True)
        fig = px.bar(
            df_plot,
            x="Year",
            y="Trade balance",
            color="Scenario",
            barmode="group",
            color_discrete_map=self.scenario_colors,
            pattern_shape="Scenario",
            pattern_shape_map={labelA: "", labelB: "/"},
            title=f"Trade balance — {fileA} vs {fileB}",
        )
        fig.update_layout(yaxis_title="billion USD")
        fig.show()
        self.save_html(fig, "Trade_Balance")

    # --- Jobs charts ---
    def show_jobs_percent(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        def to_long(df, label):
            out = df.rename(columns={"calc % of on farm jobs in FTE": "Percent"}).copy()
            out["Scenario"] = label
            return out
        dfA = to_long(self.scenarios[0]["jobs_pct"], labelA)
        dfB = to_long(self.scenarios[1]["jobs_pct"], labelB)
        df_plot = pd.concat([dfA, dfB], ignore_index=True)
        fig = px.bar(
            df_plot,
            x="Year",
            y="Percent",
            color="Scenario",
            barmode="group",
            color_discrete_map=self.scenario_colors,
            pattern_shape="Scenario",
            pattern_shape_map={labelA: "", labelB: "/"},
            title=f"Number of full time equivalent (FTE) workers — calc % of on farm jobs in FTE — {fileA} vs {fileB}",
        )
        fig.update_layout(yaxis_tickformat=",.0%")
        fig.show()
        self.save_html(fig, "Jobs_Percent_FTE")

    def show_jobs_fte_stacked(self):
        if not self.scenarios:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        fileA = self.scenarios[0]["source_path"].name
        fileB = self.scenarios[1]["source_path"].name
        labelA = self.scenarios[0]["label"]
        labelB = self.scenarios[1]["label"]
        def to_long(df, label):
            long_df = df.melt(id_vars="Year", value_vars=["jobs in crop", "jobs in livestock"], var_name="Series", value_name="Value")
            long_df["Scenario"] = label
            return long_df
        dfA = to_long(self.scenarios[0]["jobs_fte"], labelA)
        dfB = to_long(self.scenarios[1]["jobs_fte"], labelB)
        df_plot = pd.concat([dfA, dfB], ignore_index=True)
        fig = px.bar(
            df_plot,
            x="Year",
            y="Value",
            color="Series",
            barmode="stack",
            facet_col="Scenario",
            category_orders={"Series": ["jobs in crop", "jobs in livestock"]},
            title=f"Number of full time equivalent (FTE) workers — {fileA} vs {fileB}",
        )
        fig.update_layout(yaxis_title="million workers FTE")
        fig.show()
        self.save_html(fig, "Jobs_FTE_Stacked")

    # --- Cleanup ---
    def on_close(self):
        try:
            if self.session_tmp_dir.exists():
                for p in self.session_tmp_dir.iterdir():
                    try:
                        p.unlink()
                    except Exception:
                        pass
                try:
                    self.session_tmp_dir.rmdir()
                except Exception:
                    pass
        finally:
            self.root.destroy()

# --- Run dashboard ---
root = tk.Tk()
app = FableDashboard(root)
root.mainloop()

